In [11]:
import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.svm import SVR
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [17]:
X = pd.read_csv("../dataset/o_shank.csv")[:1000]
y = pd.read_csv("../dataset/distances.csv")["distance"][:1000].values
print(X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pca = PCA(n_components=10)
X_train_reduced = pca.fit_transform(X_train)
X_test_reduced = pca.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reduced)
X_test_scaled = scaler.transform(X_test_reduced)
print(X_train_scaled.shape)

(1000, 40) (1000,)
(800, 10)


In [19]:
dev = qml.device("default.qubit", wires=10)

@qml.qnode(dev)
def kernel(x1, x2, n_qubits):
    qml.AngleEmbedding(x1, wires=range(n_qubits))
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits))
    return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))


def kernel_mat(A, B):
    mat = []
    for a in A:
        row = []
        for b in B:
            row.append(kernel(a, b, n_qubits=10))
        mat.append(row)
    return np.array(mat)


def create_quantum_kernel(n_qubits, reps=2, entanglement="linear"):
    # default parameters for PauliFeatureMap ['Z', 'ZZ']
    # same as zzfeaturemap in qiskit
    feature_map = pauli_feature_map(feature_dimension=n_qubits, reps=reps, entanglement=entanglement)
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel

# Testando QSVR

## Angle Encoding

In [20]:
svr = SVR(kernel=kernel_mat)
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

RMSE: 0.19393387194753473
R2: 0.7129601766249857


## Pauli Feature Map

In [25]:
svr = SVR(kernel=create_quantum_kernel(n_qubits=10).evaluate)
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

RMSE: 0.35887138542547825
R2: 0.017092882386202546


# Testando SVR

In [24]:
svr = SVR(kernel='rbf')
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse}")
print(f"R2: {r2}")

RMSE: 0.16957338937012828
R2: 0.7805426054001287
